# 1 - Function Wrapping

Senaryo: Küçük bir "loglama araç kutusu" yapıyorsun. Herhangi bir fonksiyonu, çağrılmadan önce ve sonra ekrana bilgi yazan bir sarmalayıcıya (wrapper) geçirebilmek istiyorsun — bu tam olarak decorator'ların Bölüm 4'te yapacağı şeyin ön provası.
<p>

1. loglama_ile_calistir(fonksiyon, *args, **kwargs) adında bir HOF yaz.<br> Bu fonksiyon:<br>(a) fonksiyon 'un adını ( `fonksiyon.__name__` ) ve verilen argümanları ekrana yazsın, <br>(b) fonksiyon(*args, **kwargs) 'ı çağırıp sonucu bir değişkende
tutsun, <br>(c) sonucu ekrana yazsın, <br>(d) sonucu return etsin.<br>
2. carpan_uretici kullanarak ucle_carp = carpan_uretici(3) üret, sonra loglama_ile_calistir(ucle_carp, 7) çağır.<br>
3. bilgi_yazdir fonksiyonunu, hem positional hem keyword argümanlarla loglama_ile_calistir üzerinden çalıştır. <p>(Bilinçli yeni dokunuş: fonksiyon.__name__ — her fonksiyon nesnesinin kendi adını tutan bir özniteliği vardır; bunu görevin çözümünde kullanacaksın.)

In [9]:
# Verilen carpan_uretici ve bilgi_yazdir fonksiyonları
def carpan_uretici(katsayi):
    def carpan(sayi):
        return sayi*katsayi
    return carpan

def bilgi_yazdir(*args, **kwargs):
    return f"args={args}, kwargs{kwargs}"

# yazmamız istenen fonksiyon
def loglama_ile_calistir(fonksiyon, *args, **kwargs):
    print(f"[LOG] Fonksiyonun adı:", fonksiyon.__name__)
    print(f"[LOG] Fonksiyona verilen Argümanlar: Sıralı = {args} | Keyword ile = {kwargs}")
    sonuc = fonksiyon(*args, **kwargs)
    print("Sarılmış fonksiyonun Çıktısı: ",sonuc)
    return sonuc

ucle_carp = carpan_uretici(3)
print(ucle_carp(7))                         # Sarılmamış Hali
loglama_ile_calistir(ucle_carp, 7)          # Sarılmış Hali

print(bilgi_yazdir(1, 2, ad="Ayşe"))                    # Sarılmamış Hali
loglama_ile_calistir(bilgi_yazdir, 1, 2, ad="Ayşe")     # Sarılmış Hali

21
[LOG] Fonksiyonun adı: carpan
[LOG] Fonksiyona verilen Argümanlar: Sıralı = (7,) | Keyword ile = {}
Sarılmış fonksiyonun Çıktısı:  21
args=(1, 2), kwargs{'ad': 'Ayşe'}
[LOG] Fonksiyonun adı: bilgi_yazdir
[LOG] Fonksiyona verilen Argümanlar: Sıralı = (1, 2) | Keyword ile = {'ad': 'Ayşe'}
Sarılmış fonksiyonun Çıktısı:  args=(1, 2), kwargs{'ad': 'Ayşe'}


"args=(1, 2), kwargs{'ad': 'Ayşe'}"

# 2 - Higher Order Functions 
Senaryo: Küçük bir e-ticaret sitesi için basit bir "ürün işleme hattı (pipeline)" kuruyorsun: bir ürün listesini filtrele, dönüştür ve bir özet değere indirge.
1. [{"ad": "Kalem", "fiyat": 12, "stok": 50}, {"ad": "Defter", "fiyat": 25, "stok": 0}, {"ad": "Silgi", "fiyat": 5, "stok": 200}, {"ad": "Çanta", "fiy at": 350, "stok": 8}]
verisiyle çalış.
2. filter() ile yalnızca stokta olan ( stok > 0 ) ürünleri seç.
3. indirim_hesaplayici_uret(0.15) ile üretilen fonksiyonu map() içinde kullanarak her ürünün fiyatına %15 indirim uygula —
sonuç bir fiyat listesi olsun (ürün nesnesi değil).
4. functools.reduce() ile bu indirimli fiyatların toplamını hesapla.
5. Son adımda, en pahalı stoktaki ürünü max() ve key ile bul.

Not: Adım 3'te indirim_hesaplayici_uret 'in bir sözlük değil sayı beklediğine dikkat et — map 'e geçireceğin lambda, önce fiyatı
çekip fonksiyona vermeli.

In [22]:
from functools import reduce

def indirim_hesaplayici(indirim_orani):     # Fonksiyon Fabrikası
    def hesaplayici(fiyat):
        return fiyat * (1-indirim_orani)
    return hesaplayici

urunler = [
    {"ad":"Kalem", "fiyat":12, "stok":50},
    {"ad":"Defter", "fiyat":25, "stok":0}, 
    {"ad":"Silgi", "fiyat":5, "stok":200}, 
    {"ad":"Çanta", "fiyat":350, "stok":8}
]

stokta_olan = filter(lambda sozluk: sozluk["stok"]>0, urunler)
stokta_olan = list(stokta_olan)     # Iterator bir kere evaluate edildikten sonra silinmesin diye  

yuzde_15_indirim = indirim_hesaplayici(0.15) 

indirimli_fiyatlar = list(map(lambda sozluk: yuzde_15_indirim(sozluk["fiyat"]), stokta_olan))
print("İndirimli Fiyatlar:", indirimli_fiyatlar)

toplam_fiyat = reduce(lambda acc,x:acc+x, indirimli_fiyatlar, 0)
print(toplam_fiyat)

en_pahali_urun = max(stokta_olan, key=lambda sozluk: sozluk["fiyat"]).get("ad")
print("En pahalı ürün:",en_pahali_urun)

İndirimli Fiyatlar: [10.2, 4.25, 297.5]
311.95
En pahalı ürün: Çanta
